In [ ]:
using Bessels
include("/Users/tmalas/Desktop/Dev/Simulations_local/Code_Julia/Julia_file/classical_oscillators.jl");

In [ ]:
# const ∞ = Inf
# ħ=1
# e=1
# β = ∞ # Case zero temperature T=0, β → ∞

# #### Caracterization of the circuit #####
# E_J = 45*2*pi  # ħ * GHz
# E_Ca=92.7e-3 #h GHz
# C_a_inv=2E_Ca*2pi # hbar/e^2 GHz
# E_Cb=73.5e-3 #h GHz
# C_b_inv=2E_Cb*2pi # hbar/e^2 GHz

# ω_a = 8*2*pi
# ω_b = 5*2*pi

# Z_a=C_a_inv/ω_a # Impedance of the resonator a
# Z_b=C_b_inv/ω_b # Impedance of the resonator b

# L_a_inv=ω_a^2/C_a_inv
# L_b_inv=ω_b^2/C_b_inv

# κ=1/20; # 1/ns from Jeffrey et al.
# ϕ_quantum=1/2;

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.7*2*pi
ω_b_0 = 7.1*2*pi

Lamb_shift = - 0.05 #Lamb shift in GHz

ω_b = ω_b_0 #+ Lamb_shift*2*pi # Shifted frequency of mode b


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.2
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37*2*pi




κ_b=1/10; # 1/ns from Leon's paper

ϕ_quantum=1/2

ω_d=ω_b+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.6
g2_max = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=8*g2_max

κ_2 = 4*g2_max^2/κ_b
κ_a = 3e-3*κ_2; #from Leon's paper

In [ ]:
t_max=20/κ_b

In [ ]:
ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = two_photon_CL(t_max;
        circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ_a=0,κ_b=κ_b,ϵ_d=ϵ_d,ω_d=ω_b_0),
        init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞)

In [ ]:
Re_a_CL = ϕ_CL_a / sqrt(2*Z_a)
Im_a_CL = q_CL_a * sqrt(Z_a/2)

ϕ_rotating_a =@. ϕ_CL_a * cos(ω_a * t_list_CL) + q_CL_a*Z_a* sin(-ω_a * t_list_CL)
q_rotating_a =@. -ϕ_CL_a/Z_a * sin(-ω_a * t_list_CL) + q_CL_a* cos(ω_a * t_list_CL)

Re_rotating_a_CL = ϕ_rotating_a / sqrt(2*Z_a)
Im_rotating_a_CL = q_rotating_a * sqrt(Z_a/2)

# ϕ_rotating_b =@. ϕ_CL_b * cos(ω_a * t_list) + q_CL_b*Z_a* sin(-ω_a * t_list)
# q_rotating_b =@. -ϕ_CL_b/Z_a * sin(-ω_a * t_list) + q_CL_b* cos(ω_a * t_list);
p1=plot(Re_a_CL,Im_a_CL)
p2=plot(Re_rotating_a_CL,Im_rotating_a_CL)


plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
(Re_rotating_a_CL[end]^2+ Im_rotating_a_CL[end]^2)/(ϵ_d/2g2_max)

In [ ]:
max_t = 25
ϵ_d_list = 0.1:0.1:1
α_sq_list = Array{Float64}(undef, max_t, length(ϵ_d_list))

for i in 1:max_t
    for j in 1:length(ϵ_d_list)
        ϵ_d_i = ϵ_d_list[j]
        t_max = i/κ
        ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = two_photon_CL(t_max;
            circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ,ϵ_d=ϵ_d_i,ω_d=ω_b),
            init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞)

        ϕ_rotating_a =@. ϕ_CL_a * cos(ω_a * t_list_CL) + q_CL_a*Z_a* sin(-ω_a * t_list_CL)
        q_rotating_a =@. -ϕ_CL_a/Z_a * sin(-ω_a * t_list_CL) + q_CL_a* cos(ω_a * t_list_CL)

        Re_rotating_a_CL = ϕ_rotating_a / sqrt(2*Z_a)
        Im_rotating_a_CL = q_rotating_a * sqrt(Z_a/2)

        α_sq=(Re_rotating_a_CL[end]^2+ Im_rotating_a_CL[end]^2)
        α_sq_list[i,j] = α_sq
    end
end

In [ ]:
p1=plot(1:max_t,abs.(2*α_sq_list[:,1]./(ϵ_d_list[1]/g2_max).-1)/1, label=L"\epsilon_d="* string(round(ϵ_d_list[1]; digits = 3))*" GHz")
for j in 2:length(ϵ_d_list)
    plot!(1:max_t,abs.(2*α_sq_list[:,j]./(ϵ_d_list[j]/g2_max).-1)/1, label=L"\epsilon_d="* string(round(ϵ_d_list[j]; digits = 3))*" GHz")
end
plot!(yaxis=:log, xlabel=L"t_{max}*\kappa", ylabel=L"|2\alpha^2/(\epsilon_d/g_2)-1|")
plot(p1)
#savefig(p1, "../Figures/Two-photon/alpha_sq_vs_eps_d.pdf")

In [ ]:
max_t = 25
t_max = max_t/κ
E_J_list = 60*2*pi:.5*2*pi:65*2*pi
α_sq_list = Array{Float64}(undef, max_t, length(E_J_list))


for j in 1:length(E_J_list)
    E_J_j = E_J_list[j]
    g2_max_j = besselj(1,ϵ_0)*E_J_j*φ_a^2*φ_b # (Leon's paper)
    ϵ_d_j=0.75*g2_max_j
    ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = two_photon_CL(t_max;
        circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J = E_J_j, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ,ϵ_d=ϵ_d_j,ω_d=ω_b),
        init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞)

    ϕ_rotating_a =@. ϕ_CL_a * cos(ω_a * t_list_CL) + q_CL_a*Z_a* sin(-ω_a * t_list_CL)
    q_rotating_a =@. -ϕ_CL_a/Z_a * sin(-ω_a * t_list_CL) + q_CL_a* cos(ω_a * t_list_CL)

    Re_rotating_a_CL = ϕ_rotating_a / sqrt(2*Z_a)
    Im_rotating_a_CL = q_rotating_a * sqrt(Z_a/2)

    
    for i in 1:max_t
        t_i = i/κ
        index = findmin(abs.(t_list_CL .- t_i))[2]
        α_sq_i_j=(Re_rotating_a_CL[index]^2+ Im_rotating_a_CL[index]^2)
        α_sq_list[i,j] = α_sq_i_j
    end
end

In [ ]:
g2_max_list=besselj(1,ϵ_0).*E_J_list*φ_a^2*φ_b # (Leon's paper)
p1=plot(1:max_t,abs.(α_sq_list[:,1]./(ϵ_d/2g2_max) .-1.1), label=L"E_J/2\pi="* string(round(E_J_list[1]/2pi; digits = 1))*" GHz")
for j in 2:length(g2_max_list)
    plot!(1:max_t,abs.(α_sq_list[:,j]./(ϵ_d/2g2_max).-1.1), label=L"E_J/2\pi="* string(round(E_J_list[j]/2pi; digits = 1))*" GHz")
end
plot!(yaxis=:log, xlabel=L"t_{max}\kappa", ylabel=L"|\alpha^2/(\epsilon_d/2g_2)-1.1|", legend=:bottomleft)
plot(p1)
#savefig(p1, "../Figures/Two-photon/alpha_sq_vs_g2.pdf")

In [ ]:
α_init=0.05#0.6069077429764356
θ=-π/2:pi/10:3pi/2
Re_a_list = Vector{Vector{Float64}}(undef, length(θ))
Im_a_list = Vector{Vector{Float64}}(undef, length(θ))
i=0
t_list_all = Vector{Float64}(undef, 0)
t_max=20/κ
for θ_i in θ
    i+=1
    ϕ_a_0 = α_init * cos(θ_i)*sqrt(2*Z_a)
    q_a_0 = α_init * sin(θ_i)*sqrt(2/Z_a)

    ϕ_a, _, q_a, _, t_list = two_photon_CL(t_max;
        circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ,ϵ_d=ϵ_d,ω_d=ω_b+0.05*2*pi),
        init_mode_a=(ϕ_a_0=ϕ_a_0,q_a_0 = q_a_0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞)

    Re_a = ϕ_a / sqrt(2*Z_a)
    Im_a = q_a * sqrt(Z_a/2)

    Re_rotating_a =@. Re_a * cos(ω_a * t_list) + Im_a* sin(-ω_a * t_list)
    Im_rotating_a =@. -Re_a * sin(-ω_a * t_list) + Im_a* cos(ω_a * t_list)

    Re_a_list[i] = Re_rotating_a
    Im_a_list[i] = Im_rotating_a
    if i==1 # For the first point, we save the time list
        t_list_all = t_list
    end
end

In [ ]:
x_min = -0.7
x_max = 0.7
y_min = -1.2
y_max = 1.2

x=x_min:0.01:x_max
y=y_min:0.01:y_max

function Pseudo_V(x,y,α_sq,κ_2)
    return κ_2*((x^4+y^4)/4 + x^2*y^2/2 - (x^2-y^2)*α_sq/2)
end

α_sq = - ϵ_d/2g2_max
α_eq = sqrt(-α_sq)
κ_2 = 4*g2_max/κ
#plot(x,y, Pseudo_V.(x,y,α_sq,κ_2))

V=[Pseudo_V(x_i,y_j,α_sq,κ_2) for x_i in x for y_j in y]
x_min = [0,0]
y_min = [-α_eq,α_eq]
heatmap(x,y,reshape(V,length(y),length(x)),color=Perso_colormap)
scatter!(x_min, y_min, markers=:x, ms=10, mc=:red)

In [ ]:
θ_list = 0:pi/100:2pi
x_list=-.8:0.01:.8
p1=heatmap(x,y,reshape(V,length(y),length(x)))#scatter(Re_a_list[1][1:500:end],Im_a_list[1][1:500:end],marker_z = t_list_all[1:500:end]*κ,msw = 0)
for i in 1:length(Re_a_list)
    cmap = (0*π < θ[i] <= 1π) ? :devon : :lajolla  # or any other color gradient
    scatter!(p1,Re_a_list[i][1:500:end],Im_a_list[i][1:500:end],marker_z = t_list_all[1:500:end]*κ,msw = 0,color = cmap)
end
plot!(x_list, -tan(0*π)*x_list, color=:black, ls=:dash)
#plot!(α_eq * cos.(θ_list),α_eq * sin.(θ_list), ls=:dash, color=:red)
scatter!(x_min, y_min, markers=:x, ms=10, mc=:red)
plot(p1,legend=false, 
xlabel=L"\Re~(a)", ylabel=L"\Im(a)", 
xlims=(-1.2,1.2), ylims=(-1.2,1.2), 
colorbar=false,
size=size_px_car)
#savefig("../Figures/Two-photon/Stability_different_start_off_resonant_buffer_drive_27_06.pdf")

In [ ]:
################################################################################################################################################################################################################################################################################################################################################################
################################################################################################################################################################################################################################################################################################################################################################
################################################################################################################################################################################################################################################################################################################################################################
################################################################################################################################################################################################################################################################################################################################################################

# Heatmap on the stability of the cat states

In [ ]:
using ProgressBars
ϵ_0_list = vcat(0.01:.025:1,1:.05:0.6pi)
ϵ_d_g2_ratio_list = 0.1:0.25:16

In [ ]:
α_sq_list = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_0_list))
#α_sq_list_4_5 = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_0_list))
#α_sq_list_3_5 = Array{Float64}(undef, length(ϵ_d_g2_ratio_list), length(ϵ_0_list))
t_max=10/κ_b
for i in ProgressBar(1:length(ϵ_0_list))
    ϵ_0_i = ϵ_0_list[i]
    g2_max_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        α_th = sqrt(ϵ_d_g2_ratio_list[j]/2)
        q_a_0 = α_th* sqrt(2/Z_a)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_max_i
        ϕ_CL_a, _, q_CL_a, _, t_list_CL = two_photon_CL(t_max;
            circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0_i,κ_a=0,κ_b=κ_b,ϵ_d=ϵ_d_j,ω_d=ω_b_0),
            init_mode_a=(ϕ_a_0=0.0,q_a_0 = q_a_0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞)

        Re_rotating_a_CL_end = (ϕ_CL_a[end] * cos(ω_a * t_list_CL[end]) + q_CL_a[end]*Z_a* sin(-ω_a * t_list_CL[end])) / sqrt(2*Z_a)
        Im_rotating_a_CL_end = (-ϕ_CL_a[end]/Z_a * sin(-ω_a * t_list_CL[end]) + q_CL_a[end]* cos(ω_a * t_list_CL[end])) * sqrt(Z_a/2)

        #=index_4_5=div(4*length(t_list_CL),5)
        Re_rotating_a_CL_4_5 = (ϕ_CL_a[index_4_5] * cos(ω_a * t_list_CL[index_4_5]) + q_CL_a[index_4_5]*Z_a* sin(-ω_a * t_list_CL[index_4_5])) / sqrt(2*Z_a)
        Im_rotating_a_CL_4_5 = (-ϕ_CL_a[index_4_5]/Z_a * sin(-ω_a * t_list_CL[index_4_5]) + q_CL_a[index_4_5]* cos(ω_a * t_list_CL[index_4_5])) * sqrt(Z_a/2)

        index_3_5=div(3*length(t_list_CL),5)
        Re_rotating_a_CL_3_5 = (ϕ_CL_a[index_3_5] * cos(ω_a * t_list_CL[index_3_5]) + q_CL_a[index_3_5]*Z_a* sin(-ω_a * t_list_CL[index_3_5])) / sqrt(2*Z_a)
        Im_rotating_a_CL_3_5 = (-ϕ_CL_a[index_3_5]/Z_a * sin(-ω_a * t_list_CL[index_3_5]) + q_CL_a[index_3_5]* cos(ω_a * t_list_CL[index_3_5])) * sqrt(Z_a/2)=#

        α_sq=(Re_rotating_a_CL_end^2+ Im_rotating_a_CL_end^2)
        α_sq_list[j,i] = α_sq/(ϵ_d_g2_ratio_list[j]/2)

        #α_sq_list_4_5[j,i] = (Re_rotating_a_CL_4_5^2+ Im_rotating_a_CL_4_5^2)/(ϵ_d_g2_ratio_list[j]/2)
        #α_sq_list_3_5[j,i] = (Re_rotating_a_CL_3_5^2+ Im_rotating_a_CL_3_5^2)/(ϵ_d_g2_ratio_list[j]/2)

    end
end

In [ ]:
α_sq_list_norm =abs.(α_sq_list .-1)
#α_sq_list_norm_4_5 =abs.(α_sq_list_4_5 .-1)
#α_sq_list_norm_3_5 =abs.(α_sq_list_3_5 .-1);


In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b.*1e3
g2_lis = 0:0.1:60 .*2pi
α_sq= (ϵ_d_g2_ratio_list./2)
heatmap(g2_list/2pi,α_sq,log10.(α_sq_list_norm))
plot!(g2_lis/2pi, (κ_b*1e3./(sqrt(2)*g2_lis)).^2,label="")
plot!(g2_lis/2pi, (4*κ_b*1e3 ./g2_lis).^2,label="")
plot!(g2_lis/2pi, (5.2*κ_b*1e3 ./g2_lis).^2,label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    size=(size_px_car[1]*1.3,size_px_car[2]*1.3),
    xlims=(minimum(g2_list/2pi),maximum(g2_list/2pi)),ylims=(minimum(α_sq),maximum(α_sq)))
#savefig("../../Figures/Two-photon/alpha_sq_vs_g2_ϵ_d_start_near_minimum.pdf")

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b.*1e3
g2_lis = 0:0.1:60 .*2pi
α_sq= (ϵ_d_g2_ratio_list./2)

p1=heatmap(g2_list/2pi,α_sq,log10.(α_sq_list_norm_3_5))
plot!(g2_lis/2pi, (κ_b*1e3./(sqrt(2)*g2_lis)).^2,label="")
plot!(g2_lis/2pi, (4*κ_b*1e3 ./g2_lis).^2,label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    title="Res. buffer drive, "*L"t=15\kappa_b",
    colorbar=false)

p2=heatmap(g2_list/2pi,α_sq,log10.(α_sq_list_norm_4_5))
plot!(g2_lis/2pi, (κ_b*1e3./(sqrt(2)*g2_lis)).^2,label="")
plot!(g2_lis/2pi, (4*κ_b*1e3 ./g2_lis).^2,label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    title="Res. buffer drive, "*L"t=20\kappa_b",
    colorbar=false)

p3=heatmap(g2_list/2pi,α_sq,log10.(α_sq_list_norm))
plot!(g2_lis/2pi, (κ_b*1e3./(sqrt(2)*g2_lis)).^2,label="")
plot!(g2_lis/2pi, (4*κ_b*1e3 ./g2_lis).^2,label="")
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    title="Res. buffer drive, "*L"t=25\kappa_b",
    colorbar=false)

plot(p1,p2,p3,layout=(2,2), size=(size_px_car[1]*1.3,size_px_car[2]*1.3),
xlims=(minimum(g2_list/2pi),maximum(g2_list/2pi)),ylims=(minimum(α_sq),maximum(α_sq)))



In [ ]:
heatmap(g2_list/2pi,α_sq_list,log10.(α_sq_list_norm))
#plot!(g2_list, κ*1e3 /8 ./g2_list)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|",
    title="Resonant buffer drive",
    colorbar=false)

In [ ]:
α_sq_list_off = Array{Float64}(undef, length(E_J_list), length(ϵ_d_g2_ratio_list))
t_max=25/κ_b
for i in 1:length(E_J_list)
    E_J_i = E_J_list[i]
    g2_max_i = besselj(1,ϵ_0)*E_J_i*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(ϵ_d_g2_ratio_list)
        α_th = sqrt(ϵ_d_g2_ratio_list[j]/2)
        q_a_0 = 0#α_th* sqrt(2/Z_a)
        ϵ_d_j = ϵ_d_g2_ratio_list[j]*g2_max_i
        ϕ_CL_a, _, q_CL_a, _, t_list_CL = two_photon_CL(t_max;
            circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J = E_J_i, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0, κ_a=0, κb=κ_b,ϵ_d=ϵ_d_j,ω_d=ω_b_0-Lamb_shift*2*pi),
            init_mode_a=(ϕ_a_0=0.0,q_a_0 = q_a_0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞)

        Re_rotating_a_CL_end = (ϕ_CL_a[end] * cos(ω_a * t_list_CL[end]) + q_CL_a[end]*Z_a* sin(-ω_a * t_list_CL[end])) / sqrt(2*Z_a)
        Im_rotating_a_CL_end = (-ϕ_CL_a[end]/Z_a * sin(-ω_a * t_list_CL[end]) + q_CL_a[end]* cos(ω_a * t_list_CL[end])) * sqrt(Z_a/2)

        α_sq=(Re_rotating_a_CL_end^2+ Im_rotating_a_CL_end^2)
        α_sq_list_off[i,j] = α_sq/ϵ_d_g2_ratio_list[j]

   end
end

In [ ]:
α_sq_list_reshaped_off =reshape(abs.(α_sq_list_off*2 .-1),length(E_J_list),length(ϵ_d_g2_ratio_list));

In [ ]:
g2_list = E_J_list.*besselj(1,ϵ_0).*φ_a^2 .*φ_b.*1e3
α_list= sqrt.(ϵ_d_g2_ratio_list./2)

min_colorbar=min(minimum(log10.(α_sq_list_reshaped)),minimum(log10.(α_sq_list_reshaped_off)))
max_colorbar=max(maximum(log10.(α_sq_list_reshaped)),maximum(log10.(α_sq_list_reshaped_off)))

l = @layout [a{0.425w} b{0.575w}]

p1 = heatmap(g2_list/2pi,α_list,log10.(α_sq_list_reshaped))
#plot!(g2_list, κ*1e3 /8 ./g2_list)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    title="Resonant buffer drive",
    colorbar=false)


p2 = heatmap(g2_list./2pi,α_list,log10.(α_sq_list_reshaped_off))
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    title="Off-resonant buffer drive",
    clim = (min_colorbar, max_colorbar),
    colorbar = true,
    colorbar_title = "\n" * L"\log_{10}(|\alpha^2/\alpha^2_{th}-1|)")

plot(p1,p2,layout=l, right_margin=30Plots.mm, size=(2,1).*size_px_car, bottom_margin=10Plots.mm, left_margin=10Plots.mm)
#savefig("../Figures/Two-photon/cat_stability_mf_30_06.pdf")

In [ ]:
################################################################################################################################################################################################################################################################################################################################################################
################################################################################################################################################################################################################################################################################################################################################################
################################################################################################################################################################################################################################################################################################################################################################
################################################################################################################################################################################################################################################################################################################################################################

In [ ]:
t_max=25/κ_b
ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = two_photon_CL(t_max;
        circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J = E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ_a=κ_a,κ_b=κ_b,ϵ_d=ϵ_d,ω_d=ω_b),
        init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞)

In [ ]:
b_exp=(ϕ_CL_b/sqrt(Z_b) + 1im * q_CL_b * sqrt(Z_b))/sqrt(2)
F_b_exp, freqs_b_exp = FFT_perso(b_exp,t_list_CL);

In [ ]:
plot(freqs_b_exp, abs.(F_b_exp))
plot!(xlims=(0, 30),
ylabel="Emission spectrum", xlabel="Frequency (GHz)",
yaxis=:log10)